# Deterministic command replay

**Dynamics replay** (`rollout_vertical_dynamics`): state `(h, vz)` with lag on vertical rate; sparse `vz_sel` is **not** backfilled in extraction.

Replay-only smoothing:
- **Ramps** (~30 s) at each new `vz_sel` plateau (no full-gap ffill — that would fly descent rate through cruise).
- **Capture** on `h_sel` only where `vz_sel` is absent, capped at 2500 fpm (not ±4000).

Orange dotted = smoothed vz target; blue dashed = raw `vz_sel` plateaus.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

PKG_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PKG_ROOT))

from pipeline.opendata import route_dataset_dir
from pipeline.replay import rollout_vertical, replay_metrics, rollout_vertical_dynamics

ROUTE_NAME = 'EHAM_LSZH'
ROUTE_DIR = route_dataset_dir(ROUTE_NAME)
COMMANDS_DIR = ROUTE_DIR / 'commands'

manifest = pd.read_parquet(ROUTE_DIR / 'manifest.parquet')
FLIGHT_IDS = manifest.loc[manifest['status'] == 'done', 'flight_id'].astype(str).tolist()
print(len(FLIGHT_IDS), 'flights')

100 flights


In [2]:
metrics_path = ROUTE_DIR / 'replay' / 'replay_metrics.parquet'
if metrics_path.exists():
    summary = pd.read_parquet(metrics_path)
    display(summary.describe())
else:
    print('run: python replay_commands.py --route', ROUTE_NAME)

,rmse_ft,mae_ft,bias_ft,n
count,100.000000,100.000000,100.000000,100.000000
mean,2260.343488,1704.695482,-439.102623,4120.200000
std,978.679399,727.080758,881.934466,235.507104
min,713.018396,527.669464,-3365.636692,3675.000000
25%,1619.772038,1227.813135,-887.356959,3960.000000
50%,2135.615361,1612.806468,-342.865441,4073.500000
75%,2684.442186,2026.973234,159.062224,4205.750000
max,5579.587977,4522.851304,1290.366093,5384.000000


In [ ]:
def plot_replay(replay: pd.DataFrame, flight_id: str):
    t = replay['t_s']
    print(replay['replay_mode'].value_counts().to_string())

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                        subplot_titles=('Altitude', 'Vertical rate'),
                        row_heights=[0.65, 0.35])
    fig.add_trace(go.Scatter(x=t, y=replay['obs_altitude_ft'], mode='lines', name='observed', line=dict(color='gray')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['gen_altitude_ft'], mode='lines', name='replay', line=dict(color='red')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['cmd_alt_ft'], mode='lines', name='h_sel', line=dict(color='blue', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['obs_vertical_rate_fpm'], mode='lines', name='obs VS', line=dict(color='gray')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['gen_rocd_fpm'], mode='lines', name='replay ROCD', line=dict(color='red')), row=2, col=1)
    # if 'cmd_vz_smooth_fpm' in replay.columns:
    #     fig.add_trace(go.Scatter(x=t, y=replay['cmd_vz_smooth_fpm'], mode='lines', name='vz target (smooth)', line=dict(color='orange', dash='dot')), row=2, col=1)
    if 'cmd_vz_fpm' in replay.columns:
        fig.add_trace(go.Scatter(x=t, y=replay['cmd_vz_fpm'], mode='lines', name='vz_sel cmd', line=dict(color='blue', dash='dash')), row=2, col=1)
    m = replay_metrics(replay)
    fig.update_layout(height=700, title=f"{flight_id} | RMSE {m['rmse_ft']:.0f} ft", hovermode='x unified')
    fig.update_yaxes(title_text='ft', row=1, col=1)
    fig.update_yaxes(title_text='fpm', row=2, col=1)
    fig.update_xaxes(title_text='time [s]', row=2, col=1)
    display(fig)

FLIGHT_INDEX = 0
output = widgets.Output()
prev_btn = widgets.Button(description='← Prev')
next_btn = widgets.Button(description='Next →')
label = widgets.HTML()


def render():
    global FLIGHT_INDEX
    with output:
        clear_output(wait=True)
        fid = FLIGHT_IDS[FLIGHT_INDEX]
        cmds = pd.read_parquet(COMMANDS_DIR / f'{fid}.parquet')
        replay = rollout_vertical_dynamics(cmds, tau_vz_s=0.0, smooth_vz_ramp_s=1)
        label.value = f'<b>{FLIGHT_INDEX + 1}</b> / {len(FLIGHT_IDS)}: {fid}'
        plot_replay(replay, fid)


def go_prev(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX - 1) % len(FLIGHT_IDS)
    render()


def go_next(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX + 1) % len(FLIGHT_IDS)
    render()


prev_btn.on_click(go_prev)
next_btn.on_click(go_next)
display(widgets.HBox([prev_btn, next_btn, label]))
display(output)
render()

Output()